In [3]:
using LowLevelFEM, LinearAlgebra

In [4]:
openGeometry("boxes.geo")

In [5]:
#openPreProcessor()

In [6]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [7]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 4250)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

 27.211773 seconds (13.15 M allocations: 1.319 GiB, 1.51% gc time, 33.28% compilation time)


0

In [8]:
contact_pair = contact(u, master="master", slave="slave", cn=1e8)

Contact("slave" -> "master", 3053 candidate nodes, 534 active, G=(9159, 60114), C=(9159, 9159))

In [9]:
support = [bc_bottom, bc_top]
free = freeDoFs(U, support)

p = nothing
rc = nothing
u_it = copy(u)

old_tags = copy(contact_pair.master_element_tags)
old_G = copy(contact_pair.G)

for iter in 1:40

    updateContact!(contact_pair, u_it)

    (; G, C, g) = contact_pair

    nchanged = count(old_tags .!= contact_pair.master_element_tags)

    dG = norm(G - old_G) /
         max(norm(old_G), eps())

    println(
        "master changes = ", nchanged,
        ", dG = ", dG
    )

    old_tags = copy(contact_pair.master_element_tags)
    old_G = copy(G)

    # Penalty contact
    p  = -C * g
    rc = -G' * p
    Kc =  G' * C * G

    # Equilibrium residual and tangent
    r = K * u_it - f + rc
    A = K + Kc

    # Homogeneous Newton correction on prescribed DoFs
    Δu = vectorField(U, "body", [0, 0, 0])
    DoFs(Δu)[free] = -A[free, free] \ DoFs(r)[free]

    r0 = norm(DoFs(r)[free])

    α = 1.0
    u_trial = copy(u_it)
    r_trial = nothing

    while α > 1e-6

        u_trial = u_it + α * Δu

        updateContact!(contact_pair, u_trial)

        (; G, C, g) = contact_pair

        p_trial = -C * g
        rc_trial = -G' * p_trial

        r_trial = K * u_trial - f + rc_trial

        if norm(DoFs(r_trial)[free]) < r0
            break
        end

        α *= 0.5
    end

    u_it = copy(u_trial)
    r = r_trial

    err = α * norm(DoFs(Δu)[free]) /
          max(norm(DoFs(u_it)), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(contact_pair.active),
        ", min gap = ", minimum(contact_pair.gap_values),
        ", error = ", err,
        ", |r| = ", norm(DoFs(r)[free])
    )

    err < 1e-8 && break
end

u = u_it

master changes = 0, dG = 7.890336144931335e-15
iter = 1, α = 1.0, active = 277, min gap = -1.9324265165709637e-6, error = 0.058578308341179004, |r| = 1121.336526950155
master changes = 1, dG = 0.0949010154779114
iter = 2, α = 0.000244140625, active = 342, min gap = -1.93197030935088e-6, error = 7.746179604184844e-7, |r| = 1039.6417882873761
master changes = 0, dG = 6.491616284908395e-6
iter = 3, α = 0.00048828125, active = 381, min gap = -1.931021038013974e-6, error = 1.2283445607085062e-6, |r| = 1024.2915482605777
master changes = 0, dG = 9.538211687741631e-7
iter = 4, α = 0.0009765625, active = 408, min gap = -1.9291087060852165e-6, error = 2.5368300112543595e-6, |r| = 1023.456848819206
master changes = 0, dG = 8.280288177094006e-7
iter = 5, α = 0.0009765625, active = 427, min gap = -1.927198215258658e-6, error = 2.5448337820244866e-6, |r| = 1021.3840471706935
master changes = 0, dG = 7.965447623729172e-7
iter = 6, α = 0.0009765625, active = 436, min gap = -1.9252889453270268e-6, err

nodal VectorField
[0.0; 0.0; … ; -0.059467195923255854; 0.011519983440011684;;]

In [10]:
showDoFResults(u, name="u cont.", visible=true, factor=1)

1

In [11]:
updateContact!(contact_pair, u)
(; G, C, g) = contact_pair

p = -C * g
rc = -G' * p
showElementResults(nodesToElements(rc, onPhysicalGroup="slave"), name="p")

2

In [12]:
showElementResults(contact_pair.gap, name="gap")

3

In [ ]:
openPostProcessor()

Két vagy több párnál majd:

```Julia
contacts = ContactSet(c1, c2, c3)

updateContact!(contacts, u_it)

Kc = sum(c.G' * c.C * c.G for c in contacts)
rc = sum(c.G' * c.C * c.g for c in contacts)

r = K * u_it - f + rc
A = K + Kc
```